In [1]:
!pip install -q datasets scikit-learn joblib pandas

In [1]:
import os
import joblib
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline

class LanguageDetectorTrainer:
    def __init__(self):
        self.model = None

    def load_data(self, df):
        self.data = df

    def split_data(self):
        # Splitting the dataset for evaluation
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.data['text'], self.data['labels'], test_size=0.2, random_state=42
        )

    def train_model(self):
        print("Training model (LinearSVC + TfidfVectorizer char 2-5)...")
        self.model = make_pipeline(
            TfidfVectorizer(analyzer="char", ngram_range=(2, 5), lowercase=True),
            LinearSVC()
        )
        self.model.fit(self.X_train, self.y_train)
        print("Training complete.")

    def evaluate_model(self):
        accuracy = self.model.score(self.X_test, self.y_test)
        print(f"Validation Accuracy: {accuracy:.4f}")

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print(f"Model successfully saved to {model_path}")

D:\miniconda3\envs\emotion_classifier\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset from Hugging Face
print("Loading 'papluca/language-identification' dataset...")
ds = load_dataset("papluca/language-identification")
df = pd.DataFrame(ds['train'])

# Initialize trainer
trainer = LanguageDetectorTrainer()
trainer.load_data(df)
trainer.split_data()
trainer.train_model()
trainer.evaluate_model()

# Save the trained model locally in Colab
path = "..\saved_models\lang_detector.pkl"
os.makedirs(os.path.dirname(path), exist_ok=True)
trainer.save_model(path)

Loading 'papluca/language-identification' dataset...


Training model (LinearSVC + TfidfVectorizer char 2-5)...
Training complete.
Validation Accuracy: 0.9956
Model successfully saved to ..\saved_models\lang_detector.pkl


In [7]:
# Test predictions inside Colab
test_set = {
    "can you help me ": "en",
    "Bonjour, comment ça va?": "fr",
    "Hola, ¿cómo estás?": "es",
    "Hallo, wie geht es dir?": "de",
    "Ciao, come stai?": "it",
    "Привет, как дела?": "ru",
    "你好，你怎么样？": "zh",
    "こんにちは、お元気ですか？": "ja",
    "안녕하세요, 어떻게 지내세요?": "ko",
}

# Quick validation loop using the saved pipeline
pipeline = joblib.load("..\saved_models\lang_detector.pkl")

hit_count = 0
for text, expected_lang in test_set.items():
    predicted_lang = pipeline.predict([text])[0]
    print(f"Text: '{text}' | Predicted: {predicted_lang} | Expected: {expected_lang}")
    if predicted_lang == expected_lang:
        hit_count += 1
print(f"\nCustom Test Set Accuracy: {hit_count}/{len(test_set)} = {hit_count/len(test_set):.2%}")

Text: 'can you help me ' | Predicted: tr | Expected: en
Text: 'Bonjour, comment ça va?' | Predicted: fr | Expected: fr
Text: 'Hola, ¿cómo estás?' | Predicted: es | Expected: es
Text: 'Hallo, wie geht es dir?' | Predicted: de | Expected: de
Text: 'Ciao, come stai?' | Predicted: it | Expected: it
Text: 'Привет, как дела?' | Predicted: ru | Expected: ru
Text: '你好，你怎么样？' | Predicted: zh | Expected: zh
Text: 'こんにちは、お元気ですか？' | Predicted: ja | Expected: ja
Text: '안녕하세요, 어떻게 지내세요?' | Predicted: pl | Expected: ko

Custom Test Set Accuracy: 7/9 = 77.78%
